# Demand Rising Notebook

## Import Libraries and Dataset

In [1]:
import pandas as pd
import numpy as np


import matplotlib.pyplot as plt
import seaborn as sns


import os
from dotenv import load_dotenv
from pathlib import Path

from sqlalchemy import create_engine, text

import warnings
warnings.filterwarnings('ignore')

In [2]:
env_path = Path.cwd()/"github_repo_cloned"/"festival_restocking_problem"/".env"
load_dotenv(env_path)


host = os.getenv("DB_HOST")
user = os.getenv("DB_USER")
port = os.getenv("DB_PORT")
password = os.getenv("DB_PASSword")
db = os.getenv("DB_NAME")

In [3]:
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{db}")

def sql(query, engine = engine):
    with engine.connect() as conn:
        df = pd.read_sql(text(query),conn)
    return df

In [5]:
data = sql('select * from festival_weekly_base')

In [6]:
data.head()

,week,interest,festival,state,festival_date,wk_before,normal_interest
0,2020-05-17,1.0,Diwali,MH,2020-11-14,26.0,1.06
1,2020-05-24,1.0,Diwali,MH,2020-11-14,25.0,1.06
2,2020-05-31,1.0,Diwali,MH,2020-11-14,24.0,1.06
3,2020-06-07,1.0,Diwali,MH,2020-11-14,23.0,1.06
4,2020-06-14,1.0,Diwali,MH,2020-11-14,22.0,1.06


## Initial Data Checking

In [8]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44880 entries, 0 to 44879
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   week             44880 non-null  object 
 1   interest         44880 non-null  float64
 2   festival         44880 non-null  object 
 3   state            44880 non-null  object 
 4   festival_date    44880 non-null  object 
 5   wk_before        44880 non-null  float64
 6   normal_interest  44880 non-null  float64
dtypes: float64(3), object(4)
memory usage: 2.4+ MB


In [12]:
data[['week','festival_date']] = data[['week','festival_date']].apply(lambda x: pd.to_datetime(x))

In [13]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44880 entries, 0 to 44879
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   week             44880 non-null  datetime64[ns]
 1   interest         44880 non-null  float64       
 2   festival         44880 non-null  object        
 3   state            44880 non-null  object        
 4   festival_date    44880 non-null  datetime64[ns]
 5   wk_before        44880 non-null  float64       
 6   normal_interest  44880 non-null  float64       
dtypes: datetime64[ns](2), float64(3), object(2)
memory usage: 2.4+ MB


In [14]:
data.isnull().sum()

week               0
interest           0
festival           0
state              0
festival_date      0
wk_before          0
normal_interest    0
dtype: int64

In [16]:
data.duplicated().sum()

np.int64(0)

In [17]:
data.head()

,week,interest,festival,state,festival_date,wk_before,normal_interest
0,2020-05-17,1.0,Diwali,MH,2020-11-14,26.0,1.06
1,2020-05-24,1.0,Diwali,MH,2020-11-14,25.0,1.06
2,2020-05-31,1.0,Diwali,MH,2020-11-14,24.0,1.06
3,2020-06-07,1.0,Diwali,MH,2020-11-14,23.0,1.06
4,2020-06-14,1.0,Diwali,MH,2020-11-14,22.0,1.06


In [24]:
data[['interest','wk_before']] = data[['interest','wk_before']].apply(lambda x: x.astype(int))

In [25]:
data.head()

,week,interest,festival,state,festival_date,wk_before,normal_interest
0,2020-05-17,1,Diwali,MH,2020-11-14,26,1.06
1,2020-05-24,1,Diwali,MH,2020-11-14,25,1.06
2,2020-05-31,1,Diwali,MH,2020-11-14,24,1.06
3,2020-06-07,1,Diwali,MH,2020-11-14,23,1.06
4,2020-06-14,1,Diwali,MH,2020-11-14,22,1.06


In [28]:
data['state'].unique()

array(['MH', 'WB', 'KL', 'TN', 'UP', 'GJ', 'KA', 'PB', 'DL', 'RJ', 'AS',
       'BR', 'TG', 'AP', 'OD'], dtype=object)

In [29]:
state_names = {
    'MH': 'Maharashtra', 'WB': 'West Bengal', 'KL': 'Kerala',
    'TN': 'Tamil Nadu', 'UP': 'Uttar Pradesh', 'GJ': 'Gujarat',
    'KA': 'Karnataka', 'PB': 'Punjab', 'DL': 'Delhi',
    'RJ': 'Rajasthan', 'AS': 'Assam', 'BR': 'Bihar',
    'TG': 'Telangana', 'AP': 'Andhra Pradesh', 'OD': 'Odisha'
}

data['state'] = data['state'].map(state_names)

In [30]:
data.head()

,week,interest,festival,state,festival_date,wk_before,normal_interest
0,2020-05-17,1,Diwali,Maharashtra,2020-11-14,26,1.06
1,2020-05-24,1,Diwali,Maharashtra,2020-11-14,25,1.06
2,2020-05-31,1,Diwali,Maharashtra,2020-11-14,24,1.06
3,2020-06-07,1,Diwali,Maharashtra,2020-11-14,23,1.06
4,2020-06-14,1,Diwali,Maharashtra,2020-11-14,22,1.06


In [36]:
data = data[data['wk_before']>=0]

In [37]:
data.shape

(23115, 7)

In [38]:
data[data['wk_before'] ==0]

,week,interest,festival,state,festival_date,wk_before,normal_interest
26,2020-11-15,18,Diwali,Maharashtra,2020-11-14,0,1.06
78,2021-11-07,6,Diwali,Maharashtra,2021-11-04,0,1.09
129,2022-10-23,98,Diwali,Maharashtra,2022-10-24,0,1.11
182,2023-11-12,78,Diwali,Maharashtra,2023-11-12,0,1.11
234,2024-11-03,8,Diwali,Maharashtra,2024-10-31,0,1.11
...,...,...,...,...,...,...,...
44645,2021-04-11,49,Bihu (Rongali),Odisha,2021-04-14,0,8.34
44698,2022-04-17,20,Bihu (Rongali),Odisha,2022-04-14,0,14.66
44750,2023-04-16,12,Bihu (Rongali),Odisha,2023-04-14,0,14.57
44802,2024-04-14,28,Bihu (Rongali),Odisha,2024-04-14,0,12.79


## Data Modification

## Analysis